# MIT4 Mitral Cell Model

Interactive NEURON simulation of a 4-compartment mitral cell from **ModelDB 2487**.

Based on: Bhalla & Bower (1993) *J. Neurophysiol.* 69:1948-1983
Model implementation by Andrew Davison, The Babraham Institute.

**Compartments:** soma, glomerulus, primary dendrite, secondary dendrite + 3 linking compartments

In [2]:
from neuron import h
from neuron.units import ms, mV
import numpy as np
import os
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams['figure.dpi'] = 120
%matplotlib inline

In [3]:
# Load compiled mechanisms
mech_path = os.path.abspath('2487-master/arm64/libnrnmech.dylib')
h.nrn_load_dll(mech_path)
print('Mechanisms loaded successfully')

Mechanisms loaded successfully


## Cell Construction
The MIT4 model has 7 sections in a tree topology:

In [4]:
def build_mit4():
    """Build and return the MIT4 mitral cell model."""
    
    # --- Parameters ---
    Atotal = 100000.0    # total membrane area (um2)
    Len = 100.0          # section length (um)
    RM = 100000.0        # membrane resistivity (ohm.cm2)
    Erest = -65.0        # resting potential (mV)
    p, q, r = 0.051, 0.084, 0.328  # area fractions: soma, glom, prim
    gpg, gsp, gsd = 5.86e-5, 5.47e-5, 1.94e-4  # axial conductances (S/cm2)

    # --- Create sections ---
    sections = {}
    for name in ['soma', 'glom', 'prim', 'dend', 's2d', 's2p', 'p2g']:
        sections[name] = h.Section(name=name)
    
    soma, glom, prim, dend = sections['soma'], sections['glom'], sections['prim'], sections['dend']
    s2d, s2p, p2g = sections['s2d'], sections['s2p'], sections['p2g']

    # --- Topology (matches original HOC) ---
    s2p.connect(soma(0), 0)
    prim.connect(s2p(1), 0)
    p2g.connect(prim(1), 0)
    glom.connect(p2g(1), 0)
    s2d.connect(soma(1), 0)
    dend.connect(s2d(1), 0)

    for sec in [s2d, s2p, p2g]:
        sec.L = 1; sec.diam = 1

    # --- Compute areas ---
    Asoma = p * Atotal
    Aglom = q * Atotal
    Aprim = r * Atotal
    Adend = Atotal - Asoma - Aglom - Aprim

    def set_size(sec, area):
        sec.diam = area / (np.pi * Len); sec.L = Len

    def set_ra(sec, g):
        sec.Ra = (np.pi * 1e4) / (4 * Atotal) * (1.0 / g)

    def setup(sec, area, channels):
        sec.L = Len; sec.Ra = 1e-7; set_size(sec, area)
        for ch in channels: sec.insert(ch)
        sec.e_pas = Erest; sec.g_pas = 1.0 / RM

    # --- Soma ---
    setup(soma, Asoma,
          ['pas', 'nafast', 'kfasttab', 'kslowtab', 'kA', 'kca3', 'lcafixed', 'cad'])
    soma.gnabar_nafast = 0.1532; soma.gkbar_kfasttab = 0.1956
    soma.gkbar_kslowtab = 0.0028; soma.gkbar_kA = 0.00587
    soma.gkbar_kca3 = 0.0142; soma.gcabar_lcafixed = 0.0040
    soma.depth_cad = 8

    # --- Glomerulus ---
    setup(glom, Aglom, ['pas', 'kslowtab', 'lcafixed', 'cad'])
    glom.gkbar_kslowtab = 0.020; glom.gcabar_lcafixed = 0.0095
    glom.depth_cad = 8

    # --- Primary dendrite ---
    setup(prim, Aprim,
          ['pas', 'nafast', 'kfasttab', 'kslowtab', 'lcafixed', 'cad'])
    prim.gkbar_kfasttab = 0.00123; prim.gnabar_nafast = 0.00134
    prim.gkbar_kslowtab = 0.00174; prim.gcabar_lcafixed = 0.0022
    prim.depth_cad = 8

    # --- Secondary dendrite ---
    setup(dend, Adend, ['pas', 'kfasttab', 'nafast'])
    dend.gkbar_kfasttab = 0.0330; dend.gnabar_nafast = 0.0226

    # --- Linking compartment resistances ---
    set_ra(s2d, gsd); set_ra(s2p, gsp); set_ra(p2g, gpg)

    # --- Ion reversal potentials ---
    for sec in h.allsec():
        if sec.has_membrane('ca_ion'):
            sec.eca = 70; sec.cai = 0.00001; sec.cao = 2
        if sec.has_membrane('na_ion'):
            sec.ena = 45
        if sec.has_membrane('k_ion'):
            sec.ek = -70

    return sections

print('Cell builder defined')

Cell builder defined


In [5]:
import neuron
print(neuron.__file__)
print(neuron.__version__)

c:\nrn\lib\python\neuron\__init__.py
9.0.1


In [6]:
from neuron import h
print(h.nrnversion())

NEURON -- VERSION 9.0.1 HEAD (b12a541+) 2025-11-14


In [7]:
import os
print(os.listdir("x86_64"))

['cadecay.cpp', 'cadecay.o', 'kA.cpp', 'kA.o', 'kca3.cpp', 'kca3.o', 'kfasttab.cpp', 'kfasttab.o', 'kslowtab.cpp', 'kslowtab.o', 'lcafixed.cpp', 'lcafixed.o', 'libnrnmech.so', 'makemod2c_inc', 'mod_func.cpp', 'mod_func.o', 'nafast.cpp', 'nafast.o', 'special', 'special.nrn']


In [20]:
from neuron import h

s = h.Section()
s.insert("nafast")
print("nafast loaded successfully!")

ValueError: argument not a density mechanism name.

In [21]:
import os

print("Current directory:", os.getcwd())
print("Current dir files:", "nrnmech.dll" in os.listdir())

print("x86_64 exists:", os.path.exists("x86_64"))
if os.path.exists("x86_64"):
    print("x86_64 contents:")
    print(os.listdir("x86_64"))

Current directory: c:\Users\abhia\Downloads\2487-master_1\2487-master
Current dir files: True
x86_64 exists: True
x86_64 contents:
['cadecay.cpp', 'cadecay.o', 'kA.cpp', 'kA.o', 'kca3.cpp', 'kca3.o', 'kfasttab.cpp', 'kfasttab.o', 'kslowtab.cpp', 'kslowtab.o', 'lcafixed.cpp', 'lcafixed.o', 'libnrnmech.so', 'makemod2c_inc', 'mod_func.cpp', 'mod_func.o', 'nafast.cpp', 'nafast.o', 'special', 'special.nrn']


In [8]:
from neuron import h
import os

dll = os.path.join(os.getcwd(), "nrnmech.dll")
print("DLL path:", dll)

try:
    h.nrn_load_dll(dll)
    print("DLL loaded successfully")
except Exception as e:
    print("Load error:", e)

NEURON: The user defined name already exists: cad
 near line 0
 objref hoc_obj_[2]
                   ^
        nrn_load_dll("e:\IISER C...")


DLL path: e:\IISER CAMP 2026\Code\Project_2\2487-master\nrnmech.dll
Load error: hocobj_call error: hoc_execerror: The user defined name already exists: cad


In [9]:
from neuron import h

s = h.Section()

try:
    s.insert("nafast")
    print("SUCCESS: nafast is available")
except Exception as e:
    print("ERROR:", e)

SUCCESS: nafast is available


In [10]:
import os

for f in os.listdir("x86_64"):
    if "nrn" in f.lower() or f.endswith(".dll") or f.endswith(".so"):
        print(f)

libnrnmech.so
special.nrn


In [11]:
# Build the cell
cell = build_mit4()
print('Cell built! Topology:')
h.topology()

Cell built! Topology:

|-|       __nrnsec_00000178ad13e320(0-1)
|-|       soma(0-1)
   `|       s2d(0-1)
     `|       dend(0-1)
 `|       s2p(0-1)
   `|       prim(0-1)
     `|       p2g(0-1)
       `|       glom(0-1)



1.0

## Interactive Simulation Controls

Adjust the stimulation parameters and click **Run Simulation** to see the response.

In [12]:
# Rebuild function that returns recorders for each run
def run_mit4(Ifull, stim_location, stim_dur, stim_delay, tstop):
    """Run MIT4 simulation with given parameters.
    
    Returns: dict with tvec and voltage vectors
    """
    # Clear any existing stim
    # Build fresh cell each run to avoid state contamination
    cell = build_mit4()
    
    # Setup stimulation
    alphas, alphag = 1.37, 1.85
    Atotal = 100000.0
    injcurrdens = Ifull / 100072.0  # nA/um2
    
    sstim = gstim = None
    if stim_location in ['Soma', 'Both']:
        sstim = h.IClamp(cell['soma'](0.5))
        sstim.delay = stim_delay; sstim.dur = stim_dur
        sstim.amp = alphas * injcurrdens * Atotal
    if stim_location in ['Glomerulus', 'Both']:
        gstim = h.IClamp(cell['glom'](0.5))
        gstim.delay = stim_delay; gstim.dur = stim_dur
        gstim.amp = alphag * injcurrdens * Atotal
    
    # Setup recorders
    rec = {}
    rec['t'] = h.Vector().record(h._ref_t)
    for name in ['soma', 'glom', 'prim', 'dend']:
        rec[name] = h.Vector().record(cell[name](0.5)._ref_v)
    
    # Run simulation
    h.dt = 0.025
    h.finitialize(-65)
    while h.t < tstop:
        h.fadvance()
    
    # Convert to numpy arrays
    result = {k: np.array(v.to_python()) for k, v in rec.items()}
    return result

# Test run
result = run_mit4(Ifull=1.6, stim_location='Soma', stim_dur=15000, stim_delay=50, tstop=90)
sv = result['soma']
spikes = sum(1 for i in range(1, len(sv)) if sv[i] > 0 and sv[i-1] <= 0)
print(f'Test run: {spikes} spikes, V range: [{min(sv):.1f}, {max(sv):.1f}] mV')

Test run: 3 spikes, V range: [-65.8, 30.9] mV


In [ ]:
# Create interactive controls
stim_loc = widgets.Dropdown(
    options=['Soma', 'Glomerulus', 'Both'],
    value='Soma',
    description='Stimulus at:'
)

Ifull_slider = widgets.FloatSlider(
    value=1.6, min=0.0, max=3.0, step=0.1,
    description='Ifull (nA):',
    readout_format='.2f',
)

stim_dur = widgets.FloatSlider(
    value=100, min=5, max=500, step=5,
    description='Duration (ms):',
)

stim_delay = widgets.FloatSlider(
    value=50, min=0, max=200, step=5,
    description='Delay (ms):',
)

tstop_slider = widgets.FloatSlider(
    value=200, min=20, max=500, step=10,
    description='Tstop (ms):',
)

run_btn = widgets.Button(
    description='Run Simulation',
    button_style='primary',
    layout=widgets.Layout(width='200px')
)

out = widgets.Output()

def on_run(b):
    with out:
        clear_output(wait=True)
        print('Running simulation...', flush=True)
        try:
            result = run_mit4(
                Ifull=Ifull_slider.value,
                stim_location=stim_loc.value,
                stim_dur=stim_dur.value,
                stim_delay=stim_delay.value,
                tstop=tstop_slider.value
            )
            
            fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
            
            colors = {'soma': '#1f77b4', 'glom': '#ff7f0e', 'prim': '#2ca02c', 'dend': '#d62728'}
            
            # Top panel: all compartments
            ax = axes[0]
            for name in ['soma', 'glom', 'prim', 'dend']:
                ax.plot(result['t'], result[name], label=name, color=colors[name], lw=0.8)
            ax.axhline(0, color='gray', ls=':', lw=0.5)
            ax.set_ylabel('V (mV)')
            ax.legend(loc='upper right', ncol=4, fontsize=9)
            ax.set_title(f'MIT4 Mitral Cell Response (Ifull={Ifull_slider.value:.1f} nA, {stim_loc.value} stim)')
            
            # Bottom panel: soma only with spike markers
            ax = axes[1]
            t = result['t']; v = result['soma']
            ax.plot(t, v, color=colors['soma'], lw=1)
            
            spike_times = []
            for i in range(1, len(v)):
                if v[i] > 0 and v[i-1] <= 0:
                    spike_times.append(t[i])
            
            if spike_times:
                ax.scatter(spike_times, [25]*len(spike_times), marker='v', 
                          color='red', s=50, zorder=5)
                for st in spike_times:
                    ax.axvline(st, color='red', alpha=0.3, lw=0.5)
            
            ax.set_xlabel('Time (ms)')
            ax.set_ylabel('Soma V (mV)')
            
            stim_start = stim_delay.value
            stim_end = stim_delay.value + stim_dur.value
            ax.axvspan(stim_start, stim_end, alpha=0.08, color='green', label='Stimulus')
            
            ax.legend(fontsize=9)
            
            plt.tight_layout()
            plt.show()
            
            # Summary
            print(f'Spikes: {len(spike_times)}')
            print(f'Soma V range: [{min(v):.1f}, {max(v):.1f}] mV')
            if spike_times:
                isi = np.diff(spike_times)
                print(f'Mean ISI: {np.mean(isi):.2f} ms')
                print(f'Freq: {1000/np.mean(isi):.1f} Hz')
                
        except Exception as e:
            print(f'Error: {e}')
            import traceback
            traceback.print_exc()

run_btn.on_click(on_run)

# Layout
ui = widgets.VBox([
    widgets.HBox([stim_loc, Ifull_slider]),
    widgets.HBox([stim_delay, stim_dur, tstop_slider]),
    run_btn,
    out
])
display(ui)

## Channel Distribution

| Compartment | Channels |
|---|---|
| **Soma** | pas, nafast, kfasttab, kslowtab, kA, kca3, lcafixed, cad |
| **Glomerulus** | pas, kslowtab, lcafixed, cad |
| **Primary dendrite** | pas, nafast, kfasttab, kslowtab, lcafixed, cad |
| **Secondary dendrite** | pas, kfasttab, nafast |

## Notes

- The original model uses `FUNCTION_TABLE` for kfasttab and kslowtab, loaded from external data files. This notebook replaces those with analytic TABLE-based implementations for compatibility with NEURON 9.
- Reference: Bhalla US & Bower JM (1993) Exploring parameter space in detailed single neuron models. *J. Neurophysiol.* 69:1948-1983.
- ModelDB entry: [2487](https://senselab.med.yale.edu/ModelDB/ShowModel?model=2487)